In [1]:
# cell 1 — imports
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import StratifiedKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

PROJECT_ROOT  = Path('..')
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'coatings_manifest.csv'

df = pd.read_csv(MANIFEST_PATH)
df['abs_path'] = df['path'].apply(lambda p: PROJECT_ROOT.resolve() / p)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "— CPU only"}')
print(f'Manifest : {len(df)} images, {df["label"].nunique()} classes')

PyTorch  : 2.11.0+cpu
CUDA     : False  — CPU only
Manifest : 45 images, 7 classes


In [2]:
# cell 2 — build label ↔ index mapping
LABEL_CLASSES = sorted(df['label'].unique().tolist())
LABEL_TO_IDX  = {lbl: i for i, lbl in enumerate(LABEL_CLASSES)}
IDX_TO_LABEL  = {i: lbl for lbl, i in LABEL_TO_IDX.items()}

df['label_idx'] = df['label'].map(LABEL_TO_IDX)

print('Label → Index mapping:')
for lbl, idx in LABEL_TO_IDX.items():
    count = (df['label'] == lbl).sum()
    print(f'  {idx}  {lbl:<20s}  ({count} images)')

Label → Index mapping:
  0  adhesion_loss         (6 images)
  1  blistering            (7 images)
  2  chalking              (6 images)
  3  corrosion             (8 images)
  4  crack_paint           (6 images)
  5  fading                (6 images)
  6  sagging               (6 images)


In [3]:
# cell 3 — augmentation pipelines (copied from 02_augmentation)
IMAGE_SIZE    = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def make_train_transforms(vertical_flip_p: float = 0.3) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=IMAGE_SIZE),
        A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=vertical_flip_p),
        A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0,
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
        A.ElasticTransform(alpha=40, sigma=5, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.GaussNoise(p=0.15),
        A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(16, 32),
                        hole_width_range=(16, 32), fill=0, p=0.4),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

TRAIN_TRANSFORMS = {
    'sagging':   make_train_transforms(vertical_flip_p=0.0),
    '__default': make_train_transforms(vertical_flip_p=0.3),
}

def get_train_transform(label: str) -> A.Compose:
    return TRAIN_TRANSFORMS.get(label, TRAIN_TRANSFORMS['__default'])

val_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMAGE_SIZE),
    A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Augmentation pipelines ready')

Augmentation pipelines ready


C:\Users\gaura\AppData\Local\Temp\ipykernel_9212\753875970.py:9: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(IMAGE_SIZE, IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0),
C:\Users\gaura\AppData\Local\Temp\ipykernel_9212\753875970.py:12: UserWarning: Argument(s) 'value' are not valid for transform Rotate
  A.Rotate(limit=30, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.7),
C:\Users\gaura\mambaforge\envs\coating-cv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\gaura\AppData\Local\Temp\ipykernel_9212\753875970.py:13: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0,
C:\Users\gaura\AppData\Local\Temp\ipykernel_9212\753875970.py:36: UserWarning: Argument(s) 'value' are not vali

In [4]:
# cell 4 — PyTorch Dataset
class CoatingsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mode: str = 'train'):
        '''
        mode = 'train' → class-aware augmentation
        mode = 'val'   → resize + normalize only
        '''
        self.df   = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = cv2.imread(str(row['abs_path']))

        if image is None:
            raise FileNotFoundError(f'Could not read image: {row["abs_path"]}')

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = row['label']

        if self.mode == 'train':
            tensor = get_train_transform(label)(image=image)['image']
        else:
            tensor = val_transforms(image=image)['image']

        return tensor, row['label_idx']

# Smoke test — load one batch
smoke_ds     = CoatingsDataset(df, mode='train')
smoke_loader = DataLoader(smoke_ds, batch_size=8, shuffle=True)
batch_x, batch_y = next(iter(smoke_loader))

print(f'Batch tensor shape : {batch_x.shape}')   # expect [8, 3, 224, 224]
print(f'Batch labels       : {batch_y.tolist()}')
print(f'Tensor dtype       : {batch_x.dtype}')
print(f'Value range        : [{batch_x.min():.2f}, {batch_x.max():.2f}]')

Batch tensor shape : torch.Size([8, 3, 224, 224])
Batch labels       : [5, 3, 0, 0, 0, 1, 3, 2]
Tensor dtype       : torch.float32
Value range        : [-2.12, 2.64]


In [6]:
# cell 5 — EfficientNet-B0 with custom head
NUM_CLASSES = len(LABEL_CLASSES)

def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    '''
    EfficientNet-B0 pretrained on ImageNet.
    Backbone frozen → only the classifier head trains in phase 1.
    This is correct for very small datasets — prevents catastrophic forgetting
    of ImageNet features while adapting the head to our 7 classes.
    '''
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier head: 1280 → 7
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, num_classes)
    )

    return model

model = build_model(NUM_CLASSES, freeze_backbone=True)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model            : EfficientNet-B0')
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)')
print(f'Frozen params    : {total_params - trainable_params:,}')
print(f'Output classes   : {NUM_CLASSES}')

Model            : EfficientNet-B0
Total params     : 4,016,515
Trainable params : 8,967  (0.2%)
Frozen params    : 4,007,548
Output classes   : 7
